# MarketMind — Dataset EDA

Exploratory analysis of the `FinGPT/fingpt-forecaster-dow30-202305-202312` dataset.

Run this before training to understand the data structure, label distribution, and token length distribution.

**No GPU needed** — CPU runtime is fine.

## 0. Install Dependencies

In [ ]:
%%capture
!pip install -q datasets transformers>=4.45.0 huggingface_hub

## 1. HuggingFace Login

In [ ]:
from huggingface_hub import login
login()  # paste your HF token

## 2. Load Dataset

In [ ]:
from datasets import load_dataset
from collections import Counter

ds = load_dataset("FinGPT/fingpt-forecaster-dow30-202305-202312", split="train")
dataset = ds.train_test_split(test_size=0.1, seed=42)

print(dataset)
print("\nColumns:", dataset["train"].column_names)
print("\nTotal examples:", len(ds))

## 3. Inspect a Sample Example

In [ ]:
example = dataset["train"][0]
for key, val in example.items():
    print(f"--- {key} ---")
    print(str(val)[:600])
    print()

## 4. Label and Symbol Distribution

In [ ]:
import pandas as pd

train_df = dataset["train"].to_pandas()

print("Label distribution (train):")
print(train_df["label"].value_counts().to_string())
print()
print("Symbol distribution (train):")
print(train_df["symbol"].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

train_df["label"].value_counts().plot(kind="bar", ax=axes[0], edgecolor="black")
axes[0].set_title("Label Distribution")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

train_df["symbol"].value_counts().plot(kind="bar", ax=axes[1], edgecolor="black")
axes[1].set_title("Examples per Ticker (Dow 30)")
axes[1].set_xlabel("Ticker")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Token Length Distribution

Checks how many examples exceed `max_seq_length=2048` under the Qwen3 tokenizer. This informs whether truncation will be an issue during training.

In [ ]:
from transformers import AutoTokenizer
import re

# Load tokenizer (no GPU needed)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)

SYSTEM_PROMPT = (
    "You are a seasoned stock market analyst. Your task is to list the positive "
    "developments and potential concerns for companies based on relevant news and "
    "basic financial data from the past weeks, then make a prediction about the "
    "companies' stock price movement for the upcoming week.\n\n"
    "[Positive Developments]:\n1. ...\n\n"
    "[Potential Concerns]:\n1. ...\n\n"
    "[Prediction & Analysis]:\n..."
)

def strip_llama_tags(text):
    text = re.sub(r"\[/?INST\]", "", text)
    text = re.sub(r"<<SYS>>.*?<</SYS>>", "", text, flags=re.DOTALL)
    return text.strip()

def format_example(example):
    user = strip_llama_tags(example["prompt"])
    answer = example["answer"].strip()
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )

print("Tokenizer loaded.")

In [ ]:
# Sample 300 examples for speed (full dataset takes a few minutes)
SAMPLE_N = 300
sample = dataset["train"].select(range(min(SAMPLE_N, len(dataset["train"]))))
token_counts = [len(tokenizer.encode(format_example(ex))) for ex in sample]

print(f"Token length stats (n={len(token_counts)}):")
print(f"  Min:    {min(token_counts)}")
print(f"  Max:    {max(token_counts)}")
print(f"  Mean:   {sum(token_counts)/len(token_counts):.0f}")
print(f"  Median: {sorted(token_counts)[len(token_counts)//2]}")
print(f"  > 2048: {sum(1 for t in token_counts if t > 2048)} ({100*sum(1 for t in token_counts if t > 2048)/len(token_counts):.1f}% will be truncated)")

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(token_counts, bins=40, edgecolor="black", color="steelblue")
plt.axvline(x=2048, color="red", linestyle="--", linewidth=1.5, label="max_seq_length=2048")
plt.xlabel("Token count per example")
plt.ylabel("Number of examples")
plt.title("Token Length Distribution (FinGPT Dow30, Qwen3 tokenizer)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Verify Prompt Format

Print a fully formatted ChatML example to confirm the prompt structure looks correct before training.

In [ ]:
formatted = format_example(dataset["train"][0])
print(formatted)

## 7. Period Coverage

In [ ]:
print("Date range of training examples:")
periods = sorted(train_df["period"].unique())
print(f"  First: {periods[0]}")
print(f"  Last:  {periods[-1]}")
print(f"  Total unique periods: {len(periods)}")